# $\Pi_0^1$ projector — application-only Colab demo

The implementation is in `pi1projector.py`. This notebook does **not** run convergence studies, projection-error studies, commuting checks, or verification tests.

Public workflow:

```python
P = Pi1Projector(mesh)
Pi1_u = P(u)
```

`mesh` is an ordinary straight-sided tetrahedral NGSolve mesh. `Pi1_u` is an `HCurl(order=0)` `GridFunction` on that same mesh.


In [ ]:
# Install NGSolve in Colab if needed.
import importlib.util, subprocess, sys
if importlib.util.find_spec("ngsolve") is None:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        "ngsolve==6.2.2606", "webgui_jupyter_widgets"
    ])


## Load the projector package

Upload `pi1projector.py` when prompted. The implementation remains outside the notebook.


In [ ]:
import os
if not os.path.exists("pi1projector.py"):
    from google.colab import files
    files.upload()

from pi1projector import Pi1Projector
from ngsolve import *
from ngsolve.webgui import Draw
from netgen.occ import *


## Choose an example mesh

The projector accepts any supported straight-sided tetrahedral NGSolve mesh. For the demo, choose one of three nontrivial Netgen/OCC geometries below.

- `"fichera"`: nonconvex Fichera corner
- `"coil"`: hollow cylindrical/coil domain
- `"cavity"`: box with a spherical cavity

No `mesh.Curve(...)` call is used: the present projector targets straight-sided tetrahedral meshes.


In [ ]:
# Select one: "fichera", "coil", or "cavity"
example = "fichera"

if example == "fichera":
    outer = Box((-1,-1,-1), (1,1,1))
    cut   = Box((0,0,0), (2,2,2))
    shape = outer - cut
    maxh = 0.75

elif example == "coil":
    outer = Cylinder(Axes((0,0,-0.5), Z), r=1.0, h=1.0)
    inner = Cylinder(Axes((0,0,-0.5), Z), r=0.45, h=1.0)
    shape = outer - inner
    maxh = 0.45

elif example == "cavity":
    outer = Box((-1,-1,-1), (1,1,1))
    cavity = Sphere((0,0,0), 0.45)
    shape = outer - cavity
    maxh = 0.45

else:
    raise ValueError("example must be 'fichera', 'coil', or 'cavity'")

mesh = Mesh(OCCGeometry(shape).GenerateMesh(maxh=maxh))
print("example =", example)
print("volume tetrahedra =", len(list(mesh.Elements(VOL))))
Draw(mesh)


## User field and projection

Define a three-component field `u`, build the projector once for `mesh`, and apply it.


In [ ]:
u = CF((
    x*x + y,
    y*y + z,
    z*z + x,
))

P = Pi1Projector(mesh)
Pi1_u = P(u)

Pi1_u


## Manipulate the projected field

`Pi1_u` is an ordinary NGSolve `HCurl` `GridFunction` on the original user mesh.


In [ ]:
curl_Pi1_u = curl(Pi1_u)

# Example visualization of a Cartesian volume component on the boundary.
Draw(BoundaryFromVolumeCF(Pi1_u)[0], mesh, "Pi1_u_x")


## Reuse the same projector

The expensive operator construction is reused for every additional field on the same mesh.


In [ ]:
v = CF((y*z, z*x, x*y))
Pi1_v = P(v, name="Pi1_v")
